# Build dataset

In [1]:
# Library Imports
import os
import re
import numpy as np
import pandas as pd
from tabulate import tabulate

## Data loader from Excel file

In [2]:
# ============================================================
# FILE PATHS
# ============================================================
data_dir = os.path.join(
    os.path.dirname(os.path.abspath(__file__)) 
    if "__file__" in globals() else os.getcwd(), 
    "data"
)
files = {
    "Active":   os.path.join(data_dir, "Active_companies.xlsx"),
    "Bankrupt": os.path.join(data_dir, "Bankrupt_companies.xlsx"),
}

# ============================================================
# HELPER — CLEAN RAW DATAFRAME ON LOAD
# ============================================================
# Applied once per file immediately after reading from disk.
# Handles: unnamed index columns, incorporation year parsing,
# NACE code zero-padding, financial column type conversion.

def clean_df(df):
    df = df.copy()

    # Drop unnamed index columns Orbis sometimes exports
    df = df.drop(columns=[c for c in df.columns if c.startswith("Unnamed")])

    # Incorporation year — handle both Orbis export formats:
    #   "Date of incorporation" → datetime string → extract year
    #   "Year of incorporation" → integer directly
    if "Date of incorporation" in df.columns:
        df["Year of incorporation"] = (
            pd.to_datetime(df["Date of incorporation"], errors="coerce")
            .dt.year.astype("Int16")
        )
        df = df.drop(columns=["Date of incorporation"])
    elif "Year of incorporation" in df.columns:
        df["Year of incorporation"] = (
            pd.to_numeric(df["Year of incorporation"], errors="coerce")
            .astype("Int16")
        )

    # NACE code — preserve as zero-padded 4-character string
    # e.g. integer 711 → "0711" to avoid losing leading zeros
    nace_col = "NACE Rev. 2, core code (4 digits)"
    if nace_col in df.columns:
        df[nace_col] = df[nace_col].astype(str).str.zfill(4)

    # Financial columns — any column ending with a 4-digit year
    # converted to Float64 (nullable float, supports NaN)
    fin_cols = [c for c in df.columns if re.search(r"\d{4}$", c)]
    df[fin_cols] = df[fin_cols].apply(pd.to_numeric, errors="coerce").astype("Float64")

    return df.copy()

# ============================================================
# HELPER — PRINT DATAFRAME STRUCTURE
# ============================================================
def print_df_structure(name, df):
    rows = []
    for i, col in enumerate(df.columns):
        non_null = df[col].notna().sum()
        null     = df[col].isna().sum()
        rows.append([i, col, str(df[col].dtype), f"{non_null:,}", f"{null:,}"])

    print(f"\n{'='*60}")
    print(f"  {name}")
    print(f"{'='*60}")
    print(f"  Rows: {len(df):,}    Columns: {len(df.columns)}")
    print()
    print(tabulate(
        rows,
        headers=["#", "Column", "Dtype", "Non-Null", "Null"],
        tablefmt="github",
        numalign="right",
        stralign="left",
    ))
    mem_mb = df.memory_usage(deep=True).sum() / 1024
    print(f"\n  Memory usage: {mem_mb:.2f} MB\n")

# ============================================================
# LOAD FILES
# ============================================================
df_active   = None
df_bankrupt = None

for label, path in files.items():
    if not os.path.exists(path):
        print(f"File not found: {path}")
        continue
    try:
        df = pd.read_excel(path)
        df = clean_df(df)
        if label == "Active":
            df_active = df
            df_name   = "df_active"
        else:
            df_bankrupt = df
            df_name     = "df_bankrupt"
        print(f"Loaded: {df_name} ({os.path.basename(path)})")
        print_df_structure(df_name, df)
    except Exception as e:
        print(f"Error loading {path}: {e}")

# ============================================================
# PRESERVE RAW COPIES FOR IDEMPOTENT RE-RUNS
# ============================================================
# Processing blocks always reset from these _raw copies,
# meaning they can be re-run without reloading from disk
# and without accumulating side effects from previous runs.

if df_active is not None:
    df_active_raw = df_active.copy()

if df_bankrupt is not None:
    df_bankrupt_raw = df_bankrupt.copy()

print("✓ Raw copies saved: df_active_raw, df_bankrupt_raw")

Loaded: df_active (Active_companies.xlsx)

  df_active
  Rows: 9,100    Columns: 18

|   # | Column                            | Dtype   |   Non-Null |   Null |
|-----|-----------------------------------|---------|------------|--------|
|   0 | Company name Latin alphabet       | str     |      9,100 |      0 |
|   1 | NACE Rev. 2, core code (4 digits) | str     |      9,100 |      0 |
|   2 | P/L before tax EUR 2024           | Float64 |      8,719 |    381 |
|   3 | P/L before tax EUR 2023           | Float64 |      8,629 |    471 |
|   4 | P/L before tax EUR 2022           | Float64 |      8,315 |    785 |
|   5 | Total assets EUR 2024             | Float64 |      8,721 |    379 |
|   6 | Total assets EUR 2023             | Float64 |      8,633 |    467 |
|   7 | Total assets EUR 2022             | Float64 |      8,318 |    782 |
|   8 | Current liabilities EUR 2024      | Float64 |      8,715 |    385 |
|   9 | Current liabilities EUR 2023      | Float64 |      8,628 |    472 |
|  

## Data Preprocessing & panel data structure 

In [ ]:
# ============================================================
# 0. RESET — ensure idempotent execution on re-runs
# ============================================================

df_active   = df_active_raw.copy()
df_bankrupt = df_bankrupt_raw.copy()


# ============================================================
# 1. MERGE ACTIVE AND BANKRUPT COMPANIES
# ============================================================
# Assign binary bankruptcy label:
#   0 = active / solvent company
#   1 = bankrupt / dissolved (bankruptcy) company
# Then concatenate into one unified DataFrame.
# ignore_index=True resets row index to 0..N continuously.

df_active["Bankrupt"]   = 0
df_bankrupt["Bankrupt"] = 1

df_analysis = pd.concat([df_active, df_bankrupt], ignore_index=True)

print(f"After concat: {len(df_analysis):,} rows x {len(df_analysis.columns)} columns")
print(f"  Active:   {(df_analysis['Bankrupt'] == 0).sum():,}")
print(f"  Bankrupt: {(df_analysis['Bankrupt'] == 1).sum():,}")


# ============================================================
# 2. MAP NACE CODES TO INDUSTRY SECTIONS
# ============================================================
# Orbis exports 4-digit NACE Rev. 2 codes (e.g. 4711 = supermarkets).
# We group these into broader section-level categories using the
# first 2 digits (division level) per the official Eurostat
# NACE Rev. 2 classification structure.
#
# Sections K, O, P, Q, D, T, U were excluded upstream in Orbis
# via section letter filters and should not appear here.
# Returns "Unknown" for any unmapped code for data quality detection.

nace_section_map = {
    range(1,  4):  "A - Agriculture, forestry, fishing",
    range(5,  10): "B - Mining and quarrying",
    range(10, 34): "C - Manufacturing",
    range(36, 40): "E - Water supply, waste management",
    range(41, 44): "F - Construction",
    range(45, 48): "G - Wholesale and retail trade",
    range(49, 54): "H - Transport and storage",
    range(55, 57): "I - Accommodation and food service",
    range(58, 64): "J - Information and communication",
    range(68, 69): "L - Real estate",
    range(69, 76): "M - Professional, scientific, technical",
    range(77, 83): "N - Administrative and support services",
    range(90, 94): "R - Arts, entertainment, recreation",
    range(94, 97): "S - Other service activities",
}

def nace_to_section(code):
    """
    Maps a 4-digit NACE Rev. 2 code to its section label.
    Uses first 2 digits (division) for lookup.
    Returns 'Unknown' for unmatched codes.
    """
    try:
        division = int(str(code)[:2])
        for r, section in nace_section_map.items():
            if division in r:
                return section
        return "Unknown"
    except:
        return "Unknown"

df_analysis["NACE Section"] = (
    df_analysis["NACE Rev. 2, core code (4 digits)"]
    .apply(nace_to_section)
)

unknown_nace = df_analysis[df_analysis["NACE Section"] == "Unknown"]
if len(unknown_nace) > 0:
    print(f"\nUnknown NACE codes ({len(unknown_nace)} companies):")
    print(unknown_nace["NACE Rev. 2, core code (4 digits)"].value_counts().head(10))
else:
    print("✓ All NACE codes mapped successfully")


# ============================================================
# 3. CLEAN YEAR OF INCORPORATION
# ============================================================
# Force to numeric to handle any non-numeric Orbis export values
# (empty strings, "n.a." etc.) → converted to NaN safely.
# Int64 (nullable integer) used instead of int64 to support NaN.
# Used later to compute company age at each observation year.

df_analysis["Year of incorporation"] = pd.to_numeric(
    df_analysis["Year of incorporation"], errors="coerce"
).astype("Int64")


# ============================================================
# 4. REORDER COLUMNS — IDENTIFIERS FIRST
# ============================================================
# Place identifier and label columns at the front for readability.
# Financial time-series columns follow after.
# Cosmetic only — does not affect downstream modeling.

front_cols = [
    "Company name Latin alphabet",
    "Year of incorporation",
    "Bankrupt",
    "NACE Section",
    "NACE Rev. 2, core code (4 digits)",
]

remaining_cols = [c for c in df_analysis.columns if c not in front_cols]
df_analysis = df_analysis[front_cols + remaining_cols]


# ============================================================
# 5. RESHAPE WIDE → PANEL (LONG) FORMAT
# ============================================================
# Orbis exports in wide format: one row per company, with year-
# suffixed columns for each financial variable:
#   "Total assets EUR 2022", "Total assets EUR 2023", ...
#
# Panel (long) format is required for time-series feature
# engineering: one row per company-year combination, one column
# per financial variable.
#
# Reshape approach:
#   Step A — melt:   wide → long (one row per company × column)
#   Step B — parse:  extract year and metric name from column name
#   Step C — pivot:  long → panel (one row per company × year)

id_cols = [
    "Company name Latin alphabet",
    "Year of incorporation",
    "Bankrupt",
    "NACE Section",
    "NACE Rev. 2, core code (4 digits)",
]

fin_cols = [c for c in df_analysis.columns if c not in id_cols]

# ── Step A: Melt wide → long ─────────────────────────────────
# Result: one row per (company × financial column)
# col_raw = original column name e.g. "Total assets EUR 2022"
# value   = financial figure in thousands EUR
df_long = df_analysis.melt(
    id_vars=id_cols,
    value_vars=fin_cols,
    var_name="col_raw",
    value_name="value",
)

# ── Step B: Parse year and metric name ───────────────────────
# Year:   extract trailing 4-digit year from column name
# Metric: strip "EUR YYYY" suffix to get clean variable name
df_long["Year"] = (
    df_long["col_raw"]
    .str.extract(r"(\d{4})$")
    .astype("Int64")
)

df_long["Metric"] = (
    df_long["col_raw"]
    .str.replace(r"\s*EUR\s*\d{4}$", "", regex=True)
    .str.strip()
)

df_long = df_long.drop(columns="col_raw")

# ── Step C: Pivot long → panel ───────────────────────────────
# Result: one row per (company, year)
# Each metric becomes its own column
# aggfunc="first" handles any accidental duplicates
df_panel = (
    df_long
    .pivot_table(
        index=id_cols + ["Year"],
        columns="Metric",
        values="value",
        aggfunc="first"
    )
    .reset_index()
)
df_panel.columns.name = None


# ============================================================
# 6. DROP FULLY EMPTY ROWS
# ============================================================
# After reshaping, rows exist for every company × year pair even
# when Orbis had no financial data for that year (all NaN values).
# Pre-2016 data is nearly entirely missing — these empty rows
# add no information and inflate memory usage unnecessarily.
#
# Strategy: drop rows where ALL financial metric columns are NaN.
# Rows with partial data are retained — XGBoost handles missing
# values natively so partial observations are still useful.

metric_cols = [c for c in df_panel.columns if c not in id_cols + ["Year"]]

before = len(df_panel)
df_panel = df_panel.dropna(subset=metric_cols, how="all").reset_index(drop=True)
after  = len(df_panel)

print(f"\nDropped {before - after:,} fully empty rows")
print(f"df_panel: {len(df_panel):,} rows x {len(df_panel.columns)} columns")
print(f"  Years covered: {df_panel['Year'].min()} – {df_panel['Year'].max()}")
print(f"  Active rows:   {(df_panel['Bankrupt'] == 0).sum():,}")
print(f"  Bankrupt rows: {(df_panel['Bankrupt'] == 1).sum():,}")


# ============================================================
# 7. COMPUTE COMPANY AGE AT OBSERVATION YEAR
# ============================================================
# Company age = observation year − year of incorporation.
# This gives the company's age in years at the time each
# financial statement was filed — not today's age.
#
# Example: company incorporated 1997, observation year 2022
#          → company age = 25 years at time of filing
#
# Using the observation Year column (panel year) rather than
# current year ensures age is correct for each row in the panel.
# A company with data in 2022, 2023, and 2024 will show ages
# 25, 26, and 27 respectively — reflecting its actual age when
# each set of financials was reported.
#
# Int64 used to support NaN for the one company missing
# Year of incorporation in the raw Orbis export.

df_panel["Company Age"] = (
    df_panel["Year"] - df_panel["Year of incorporation"]
).astype("Int64")

# Sanity checks
age_min = df_panel["Company Age"].min()
age_max = df_panel["Company Age"].max()
age_neg = (df_panel["Company Age"] < 0).sum()
age_null = df_panel["Company Age"].isna().sum()

print(f"Company Age range:     {age_min} – {age_max} years")
print(f"Negative ages (error): {age_neg}")   # should be 0
print(f"Missing ages:          {age_null}")  # should be ≤ 1

# Move Company Age next to Year of incorporation for readability
cols = df_panel.columns.tolist()
year_inc_idx = cols.index("Year of incorporation")
cols.remove("Company Age")
cols.insert(year_inc_idx + 1, "Company Age")
df_panel = df_panel[cols]

print_df_structure("df_panel", df_panel)

After concat: 10,236 rows x 19 columns
  Active:   9,100
  Bankrupt: 1,136
✓ All NACE codes mapped successfully

Dropped 0 fully empty rows
df_panel: 27,965 rows x 11 columns
  Years covered: 2022 – 2024
  Active rows:   25,691
  Bankrupt rows: 2,274
Company Age range:     0 – 119 years
Negative ages (error): 0
Missing ages:          0

  df_panel
  Rows: 27,965    Columns: 12

|   # | Column                            | Dtype   |   Non-Null |   Null |
|-----|-----------------------------------|---------|------------|--------|
|   0 | Company name Latin alphabet       | str     |     27,965 |      0 |
|   1 | Year of incorporation             | Int64   |     27,965 |      0 |
|   2 | Company Age                       | Int64   |     27,965 |      0 |
|   3 | Bankrupt                          | int64   |     27,965 |      0 |
|   4 | NACE Section                      | str     |     27,965 |      0 |
|   5 | NACE Rev. 2, core code (4 digits) | str     |     27,965 |      0 |
|   6 | Yea

# Feature Engineering

In [4]:
# ============================================================
# 8. FEATURE ENGINEERING
# ============================================================
# Derives financial ratios, distress indicators, and trend
# features from the 5 raw Orbis variables. All engineered
# features are prefixed with "FE_" to clearly distinguish them
# from raw Orbis variables throughout the pipeline.
#
# Features computed:
#   FE_total_liabilities  — derived balance sheet item
#   FE_non_current_assets — derived balance sheet item
#   FE_debt_ratio         — solvency / leverage
#   FE_current_ratio      — short-term liquidity
#   FE_roa                — profitability relative to assets
#   FE_negative_equity    — binary distress flag
#   FE_negative_pl        — binary distress flag
#   FE_assets_yoy_growth  — year-over-year asset trajectory
#   FE_equity_yoy_change  — year-over-year equity trajectory
#
# Division safety:
#   Where a denominator is effectively zero (abs < 0.01,
#   i.e. less than €10 in thousands EUR), the ratio is set
#   to NaN rather than using epsilon inflation. This is more
#   honest — a near-zero denominator means the ratio is
#   undefined, not astronomically large. XGBoost handles
#   NaN natively so no information is lost.
#
# Trend features:
#   YoY features are NaN for a company's earliest observed
#   year — no prior year available to diff against. Expected
#   and handled natively by XGBoost.



# Work on a copy — df_panel remains unchanged
df_features = df_panel.copy()
df_features = df_features.sort_values(
    ["Company name Latin alphabet", "Year"]
).reset_index(drop=True)

# Safe division helper — returns NaN when denominator is
# effectively zero rather than inflating with epsilon
def safe_divide(numerator, denominator, threshold=0.01):
    """
    Divides numerator by denominator element-wise.
    Returns NaN where abs(denominator) < threshold.
    threshold=0.01 corresponds to €10 in thousands EUR —
    any denominator below this is treated as undefined.
    Converts Float64 (pandas nullable) to float64 (numpy)
    before comparison to avoid pd.NA boolean ambiguity.
    Returns pandas Series to preserve index alignment.
    """
    # Convert to numpy float64 — resolves pd.NA boolean ambiguity
    # that occurs with pandas Float64 nullable dtype
    num = numerator.astype("float64")
    den = denominator.astype("float64")

    result = np.where(
        den.abs() < threshold,
        np.nan,
        num / den
    ).astype(float)

    return pd.Series(result, index=numerator.index)
# ── A: Derived balance sheet items ───────────────────────────
# Accounting identity: Assets = Equity + Liabilities
# Total liabilities was not directly exported from Orbis —
# derived here from the two available balance sheet components.

df_features["FE_total_liabilities"] = (
    df_features["Total assets"]
    - df_features["Shareholders funds"]
)

df_features["FE_non_current_assets"] = (
    df_features["Total assets"]
    - df_features["Current assets"]
)

# ── B: Solvency / leverage ────────────────────────────────────
# Proportion of total assets financed by debt.
# Values > 1 occur when equity is negative — itself a distress
# signal captured separately by FE_negative_equity flag.
# NaN when total assets is effectively zero.

df_features["FE_debt_ratio"] = safe_divide(
    df_features["FE_total_liabilities"],
    df_features["Total assets"]
)

# ── C: Liquidity ──────────────────────────────────────────────
# Current ratio < 1: current liabilities exceed current assets
# meaning the company cannot cover short-term obligations from
# liquid assets alone. Classic early warning indicator.
# NaN when current liabilities is effectively zero — a company
# with no current liabilities is not distressed, but the ratio
# is undefined rather than infinite.

df_features["FE_current_ratio"] = safe_divide(
    df_features["Current assets"],
    df_features["Current liabilities"]
)

# ── D: Profitability ──────────────────────────────────────────
# Return on assets: profit generated per unit of total assets.
# Equivalent to Altman Z-score variable X3.
# Strongest single predictor in most bankruptcy studies.
# Negative ROA = loss-making relative to asset base.
# NaN when total assets is effectively zero.

df_features["FE_roa"] = safe_divide(
    df_features["P/L before tax"],
    df_features["Total assets"]
)

# ── E: Binary distress flags ──────────────────────────────────
# Hard threshold indicators of specific danger zones.
# Int64 (nullable integer) used — Float64 columns contain
# pd.NA which cannot be cast to plain int.
#
# FE_negative_equity: shareholders funds < 0 means accumulated
#   losses have exceeded paid-in capital — the company is
#   technically insolvent under Finnish Companies Act (OYL
#   chapter 20 section 1). A board is legally required to
#   take action when equity falls below half of share capital.
#
# FE_negative_pl: company recorded a pre-tax loss this year.
#   One loss year may be recoverable — in combination with
#   negative equity and high debt ratio it signals severe
#   multi-dimensional distress.

df_features["FE_negative_equity"] = (
    (df_features["Shareholders funds"] < 0).astype("Int64")
)

df_features["FE_negative_pl"] = (
    (df_features["P/L before tax"] < 0).astype("Int64")
)

# ── F: Year-over-year trend features ─────────────────────────
# Capture trajectory of financial deterioration across the
# 2022–2024 panel window.
#
# groupby company name ensures diff/pct_change only computes
# across years of the same company — never across companies.
#
# FE_assets_yoy_growth:
#   Proportional asset change relative to prior year.
#   Uses safe_divide to handle near-zero prior year asset base.
#   Negative growth = shrinking asset base, distress signal.
#
# FE_equity_yoy_change:
#   Absolute change in shareholders funds.
#   Absolute rather than proportional — pct_change is unstable
#   when base is near zero or negative, common in distress.
#   Negative = equity eroding, strong bankruptcy predictor.

grp = df_features.groupby("Company name Latin alphabet")

# Prior year values for trend calculation
prior_assets = grp["Total assets"].transform(lambda x: x.shift(1))
prior_equity  = grp["Shareholders funds"].transform(lambda x: x.shift(1))

df_features["FE_assets_yoy_growth"] = safe_divide(
    df_features["Total assets"] - prior_assets,
    prior_assets
)

df_features["FE_equity_yoy_change"] = (
    df_features["Shareholders funds"] - prior_equity
)

# ── Validation ────────────────────────────────────────────────
fe_cols = [c for c in df_features.columns if c.startswith("FE_")]

print(f"Feature engineered columns added: {len(fe_cols)}")

print(f"\nNull counts per FE column:")
null_counts = df_features[fe_cols].isnull().sum()
for col, count in null_counts.items():
    pct = count / len(df_features) * 100
    print(f"  {col:<30} {count:>6,}  ({pct:.1f}%)")

print(f"\nDistress flag prevalence:")
print(f"  FE_negative_equity: {df_features['FE_negative_equity'].sum():,} "
      f"({df_features['FE_negative_equity'].mean():.1%})")
print(f"  FE_negative_pl:     {df_features['FE_negative_pl'].sum():,} "
      f"({df_features['FE_negative_pl'].mean():.1%})")

print(f"\nBankrupt vs Active — mean FE feature values:")
print(
    df_features.groupby("Bankrupt")[fe_cols]
    .mean()
    .round(3)
    .T
    .rename(columns={0: "Active (mean)", 1: "Bankrupt (mean)"})
    .to_string()
)

print_df_structure("df_features", df_features)

Feature engineered columns added: 9

Null counts per FE column:
  FE_total_liabilities               37  (0.1%)
  FE_non_current_assets              24  (0.1%)
  FE_debt_ratio                      37  (0.1%)
  FE_current_ratio                  130  (0.5%)
  FE_roa                             49  (0.2%)
  FE_negative_equity                 14  (0.1%)
  FE_negative_pl                     35  (0.1%)
  FE_assets_yoy_growth           10,147  (36.3%)
  FE_equity_yoy_change           10,138  (36.3%)

Distress flag prevalence:
  FE_negative_equity: 2,374 (8.5%)
  FE_negative_pl:     7,109 (25.5%)

Bankrupt vs Active — mean FE feature values:
Bankrupt              Active (mean) Bankrupt (mean)
FE_total_liabilities    5205023.473      1765006.71
FE_non_current_assets   5019062.717       630128.15
FE_debt_ratio                 0.559           1.185
FE_current_ratio              5.091           1.457
FE_roa                        0.093          -0.161
FE_negative_equity            0.059           

# Taking a look at the data & saving for XGboost model

In [5]:
# Display a sample of the engineered feature DataFrame to visually inspect the new features and their values across a subset of companies and years.

from IPython.display import HTML, display

sample = df_features.sample(10, random_state=1)
html = sample.to_html(index=False)

display(HTML(
    f'<div style="max-height:400px; overflow:auto; white-space:nowrap;">{html}</div>'
))

Company name Latin alphabet,Year of incorporation,Company Age,Bankrupt,NACE Section,"NACE Rev. 2, core code (4 digits)",Year,Current assets,Current liabilities,P/L before tax,Shareholders funds,Total assets,FE_total_liabilities,FE_non_current_assets,FE_debt_ratio,FE_current_ratio,FE_roa,FE_negative_equity,FE_negative_pl,FE_assets_yoy_growth,FE_equity_yoy_change
SUOMEN EKURIIRIT OY,2023,1,0,H - Transport and storage,4941,2024,122204.0,83498.0,48444.0,38706.0,122204.0,83498.0,0.0,0.683267,1.463556,0.396419,0,0,NaN,<NA>
LS-TEK OY,2021,3,0,F - Construction,4322,2024,109000.0,17000.0,57000.0,101000.0,118000.0,17000.0,9000.0,0.144068,6.411765,0.483051,0,0,0.710145,40000.0
ELEBET OY,1992,30,0,C - Manufacturing,2361,2022,1788000.0,927000.0,280000.0,1108000.0,2378000.0,1270000.0,590000.0,0.534062,1.928803,0.117746,0,0,NaN,<NA>
CPE SUOJASAUMA OY,2011,13,0,C - Manufacturing,1392,2024,810000.0,191000.0,26000.0,676000.0,866000.0,190000.0,56000.0,0.219400,4.240838,0.030023,0,0,-0.026966,27000.0
RAKENNUS-NELOSET OY,2016,8,0,F - Construction,4120,2024,3211000.0,1504000.0,551000.0,1729000.0,3234000.0,1505000.0,23000.0,0.465368,2.134973,0.170377,0,0,-0.065858,56000.0
PRONOR CONTROL OY,2005,18,0,"M - Professional, scientific, technical",7112,2023,433797.0,377428.0,55550.0,79119.0,606051.0,526932.0,172254.0,0.869452,1.149350,0.091659,0,0,0.561987,55119.0
BOLA LIIKUNTA- JA HYVINVOINTIKESKUS OY,2020,3,1,"R - Arts, entertainment, recreation",9311,2023,489000.0,564000.0,-64000.0,-145000.0,576000.0,721000.0,87000.0,1.251736,0.867021,-0.111111,1,1,0.321101,-84666.0
TRIKA OY,2005,17,0,I - Accommodation and food service,5610,2022,130000.0,78000.0,20000.0,229000.0,306000.0,77000.0,176000.0,0.251634,1.666667,0.065359,0,0,NaN,<NA>
OTERAN OY,2009,13,0,"M - Professional, scientific, technical",7112,2022,8538000.0,5284000.0,761000.0,4554000.0,11540000.0,6986000.0,3002000.0,0.605373,1.615821,0.065945,0,0,NaN,<NA>
PROJEKTIRAKENTAJAT JK OY,2015,7,0,F - Construction,4120,2022,701836.0,130926.0,158096.0,419297.0,753277.0,333980.0,51441.0,0.443369,5.360555,0.209878,0,0,NaN,<NA>


In [6]:
# ============================================================
# 9. SAVE FEATURE-ENGINEERED DATASET
# ============================================================
# Saves the final feature-engineered panel to CSV for use in
# the modeling notebook. Separating data preparation from
# modeling ensures:
#   - Notebook 2 can be re-run without reprocessing raw data
#   - The exact modeling input is versioned and reproducible
#   - Long Orbis Excel loading time is not repeated each run
#
# File is saved to the data/ directory alongside raw inputs.
# Float precision=6 retains sufficient decimal places for
# financial ratios without inflating file size.


output_path = os.path.join(data_dir, "final_panel.csv")

df_features.to_csv(output_path, index=False, float_format="%.6f")

print(f"Saved: {output_path}")
print(f"  Rows:    {len(df_features):,}")
print(f"  Columns: {len(df_features.columns)}")
print(f"  Size:    {os.path.getsize(output_path) / 1024**2:.2f} MB")


# ============================================================
# 10. EXPORT AGGREGATE DATA SUMMARIES FOR THE DASHBOARD
# ============================================================
# Writes column-level summaries (types, missing values, medians)
# to results/ for the Streamlit app. Only aggregates are saved —
# no company names or individual rows — so these files can be
# published while the Orbis data itself stays private.

results_dir = os.path.join(os.path.dirname(data_dir), "results")
os.makedirs(results_dir, exist_ok=True)

# ── A: Raw export — missing values by group ─────────────────
raw_summary = pd.DataFrame({
    "Column":               df_active_raw.columns,
    "Dtype":                df_active_raw.dtypes.astype(str).values,
    "Active non-null":      df_active_raw.notna().sum().values,
    "Active missing %":     (df_active_raw.isna().mean() * 100).round(1).values,
    "Bankrupt non-null":    df_bankrupt_raw.notna().sum().values,
    "Bankrupt missing %":   (df_bankrupt_raw.isna().mean() * 100).round(1).values,
})
raw_summary.to_csv(os.path.join(results_dir, "raw_columns_summary.csv"), index=False)

# ── B: Model-ready panel — missing values and group medians ─
num_cols = df_features.select_dtypes("number").columns
medians  = (
    df_features.groupby("Bankrupt")[num_cols]
    .median()
    .T
    .rename(columns={0: "Median active", 1: "Median bankrupt"})
)
panel_summary = pd.DataFrame({
    "Column":    df_features.columns,
    "Dtype":     df_features.dtypes.astype(str).values,
    "Non-null":  df_features.notna().sum().values,
    "Missing %": (df_features.isna().mean() * 100).round(1).values,
}).merge(medians, left_on="Column", right_index=True, how="left")

# Identifier-like columns: medians are not meaningful
no_median = ["Bankrupt", "Year", "Year of incorporation"]
panel_summary.loc[panel_summary["Column"].isin(no_median),
                  ["Median active", "Median bankrupt"]] = np.nan
panel_summary.to_csv(os.path.join(results_dir, "panel_columns_summary.csv"), index=False)

# ── C: Headline counts ──────────────────────────────────────
overview = pd.DataFrame({
    "Metric": ["Active companies", "Bankrupt companies", "Total companies",
               "Panel rows (company-years)", "First year", "Last year"],
    "Value":  [len(df_active_raw), len(df_bankrupt_raw),
               len(df_active_raw) + len(df_bankrupt_raw),
               len(df_features), int(df_features["Year"].min()),
               int(df_features["Year"].max())],
})
overview.to_csv(os.path.join(results_dir, "dataset_overview.csv"), index=False)

print("Saved to results/: raw_columns_summary.csv, panel_columns_summary.csv, dataset_overview.csv")

Saved: c:\A_Omat jutut\Ohjelmointi\bankruptcy_prediction\data\final_panel.csv
  Rows:    27,965
  Columns: 21
  Size:    5.65 MB
Saved to results/: raw_columns_summary.csv, panel_columns_summary.csv, dataset_overview.csv
